# Triage Agent — Walkthrough

Three scenarios run end-to-end through the full pipeline:

1. `impossible_travel` under default tier policy — clean run with grounded verdict
2. `impossible_travel` against stale threat intel — the agent does not treat stale-clean reputation as benign
3. `ransomware` P0 escalating to T3 Opus with self-consistency sampling

**How the runs are wired.** The notebook runs the real plan resolution, plan-gated fan-out, reasoning wrapper, escalation, and validator code. The sample alerts are constructed directly as `CanonicalAlertEvent` objects; LLM outputs are replayed via `SequenceClient` (responses returned in call order) so the bundle's non-deterministic retrieval IDs and timestamps don't break replay. No Anthropic API key required.

**What you'll see.**

- §1: Tier policy refuses warm `historical` context on the default hot-only path, and the verdict still validates against available evidence.
- §2: Stale/unknown threat intel is carried through reasoning and validation without supporting a false-positive verdict.
- §3: A P0 ransomware alert triggers T3 escalation, with illustrative self-consistency voting plus one replayed live Opus response.

**One real LLM call lives in the repo.** Scenario 3 replays a T3 response captured against the live Anthropic API on 2026-06-16T04:29:49Z (model `claude-opus-4-7`, 2296 prompt + 2016 completion tokens, 24.4s latency, $0.18564). The fixture is `fixtures/llm_replays/cd8a1be0d7d1e45f1148e61c.json` with `live_api: true` and `captured_at` markers. The capture script is `scripts/capture_t3_fixture.py`.

In [1]:
from datetime import UTC, datetime
import json

from triage.enrichment.base import SourceQuery
from triage.enrichment.fanout import build_default_registry, run_fanout
from triage.llm.client import LLMResponse, SequenceClient
from triage.reasoning.agent import reason
from triage.reasoning.escalation import escalate_to_t3, should_escalate
from triage.schemas.alert import Asset, CanonicalAlertEvent, Observable
from triage.schemas.plan_loader import PlanTemplateRegistry
from triage.schemas.verdict import AIMetadata
from triage.validation.validator import validate_response

registry = PlanTemplateRegistry()
sources = build_default_registry()
print(f'plan templates loaded: {registry.families()}')
print(f'sources registered: {sorted(sources.keys())}')

plan templates loaded: ['impossible_travel', 'ransomware', 'c2_callback', 'dns_exfil', 'privilege_escalation']
sources registered: ['asset_cmdb', 'historical', 'identity_store', 'log_search', 'runbook', 'threat_intel']


## 1. impossible_travel under default tier policy

The default `impossible_travel` plan has `tier_preference: ['hot']`. The `historical` source lives on warm tier, so the fan-out refuses it on policy — `enrichments_failed: ['historical']` with `('historical', 'rejected')`. That's deliberate cost discipline, not a bug: the engine documents the refusal explicitly, the reasoning step sees the missing source in the prompt, and the verdict still emits with `validation failures: 0`.

### Build the canonical alert.


In [2]:
alert = CanonicalAlertEvent(
    tenant_id='tenant_a',
    alert_id='nb_demo_01',
    source_system='okta',
    source_adapter_version='okta_v1',
    rule_id='okta.impossible_travel.v3',
    rule_family='impossible_travel',
    received_at=datetime(2026, 6, 17, 14, 32, 11, tzinfo=UTC),
    detected_at=datetime(2026, 6, 17, 14, 32, 10, tzinfo=UTC),
    severity_hint='P1',
    primary_assets=[Asset(asset_id='u_acct_lead', asset_type='user', tenant_id='tenant_a')],
    observables=[Observable(observable_type='ip', value='198.51.100.42', source_field_path='client.ipAddress')],
    summary='Sign-on from Bulgaria 30s after Portland session',
)
plan = registry.build_plan(alert.rule_family, alert.severity_hint)
print(f'plan tier_preference: {plan.tier_preference}')
print(f'plan required_sources: {plan.required_sources}')
print(f'plan optional_sources: {plan.optional_sources}')

plan tier_preference: ['hot']
plan required_sources: ['identity_store', 'historical']
plan optional_sources: ['asset_cmdb', 'threat_intel']


### Resolve the plan and run plan-gated fan-out.


In [3]:
query = SourceQuery(
    tenant_id=alert.tenant_id,
    alert_id=alert.alert_id,
    entity_id=alert.grouping_entity(),
    ioc=alert.primary_ioc(),
    extra={'rule_family': alert.rule_family},
)
bundle = run_fanout(plan, query, sources)
print(f'retrievals: {len(bundle.retrievals)}')
print(f'enrichments_failed: {bundle.enrichments_failed}')
print(f'spans: {[(s["source_type"], s["outcome"]) for s in bundle.spans]}')

retrievals: 4
enrichments_failed: ['historical']
spans: [('identity_store', 'ok'), ('asset_cmdb', 'ok'), ('threat_intel', 'ok'), ('historical', 'rejected')]


### Reason with a replayed T2 response, then validate.


In [4]:
identity_ref = bundle.by_source('identity_store')[0]
verdict_json = {
    'verdict': 'likely_true_positive',
    'confidence': 0.82,
    'severity': 'P1',
    'severity_rationale': 'Geo anomaly with intact MFA.',
    'summary': 'User u_acct_lead authenticated from Bulgaria 30s after Portland session.',
    'attack_chain': [],
    'observed_facts': [{
        'fact_id': 'f1', 'claim': 'User has MFA enabled.',
        'retrieval_id': identity_ref.retrieval_id,
        'field_path': 'mfa_enabled', 'expected_value': True, 'confidence': 0.95,
    }],
    'inferences': [{'inference_id': 'i1', 'claim': 'MFA evidence reduces credential-stuff probability.', 'supported_by_fact_ids': ['f1'], 'confidence': 0.78}],
    'recommendations': [{'priority': 1, 'action': 'force_password_reset', 'rationale': 'Defense in depth.', 'supported_by_inference_ids': ['i1'], 'blast_radius': 'low', 'reversible': True, 'automatable': False}],
    'blast_radius': {'affected_assets': ['u_acct_lead']},
    'uncertainty': {'missing_enrichments': []},
}
client = SequenceClient([LLMResponse(content=json.dumps(verdict_json), stop_reason='end_turn', tokens_in=2000, tokens_out=600, cost_usd=0.022, model='claude-sonnet-4-6')])
response, augmented, extensions = reason(alert, plan, bundle, client, sources=sources)
outcome = validate_response(response.content, augmented, triage_id='triage_nb_01', tenant_id=alert.tenant_id, alert_id=alert.alert_id, investigation_plan_dump=plan.model_dump(), received_at=alert.received_at, ai_metadata=AIMetadata(route_tier='standard_t2', model_chain=['sonnet'], cost_usd=0.022))
print(f'verdict: {outcome.verdict.verdict}')
print(f'confidence: {outcome.verdict.confidence}')
print(f'recommendations[0]: {outcome.verdict.recommendations[0].action}')
print(f'validation failures: {len(outcome.failures)}')
print(f'plan extensions: {extensions}')

verdict: likely_true_positive
confidence: 0.82
recommendations[0]: force_password_reset
validation failures: 0
plan extensions: []


## 2. Stale threat-intel does not prove benign

tenant_a sees the alert IP `198.51.100.42` as `reputation: unknown` with `provider_confidence: 0.45` and `cached_at` 90 days ago. Stale/unknown threat-intel reputation is precisely the failure mode the agent must refuse to anchor on: an attacker IP that happened to be quiet when the threat-intel feed last scanned it can sit in a "clean" cache for weeks while the attacker is active.

Below, the cell inspects the staleness signature in the retrieval, then runs `reason()` and `validate_response()` against the same bundle. The replayed T2 response demonstrates how the pipeline carries the stale threat-intel context through reasoning and validation without letting it support a false-positive verdict — the verdict lands at `likely_true_positive` with `threat_intel_fresh` listed in `uncertainty.missing_enrichments`.

In [5]:
# Inspect the staleness signature in the threat_intel retrieval.
ti_refs = bundle.by_source('threat_intel')
ref = ti_refs[0]
print(f'provider:            {ref.provider}')
print(f'provider_confidence: {ref.provider_confidence}')
print(f'cached_at:           {ref.cached_at}')
print(f'reputation:          {ref.payload.get("reputation")}')
print()

# Run reason() against the same bundle with a canned T2 response that
# correctly handles the stale signal: cites the threat_intel staleness in
# uncertainty.missing_enrichments and refuses to anchor a likely_false_positive
# verdict on the cached "unknown" reputation. The reasoning, validator, and
# audit ledger all execute on real SUT code; only the model output is replayed.
ti_ref = bundle.by_source('threat_intel')[0]
id_ref = bundle.by_source('identity_store')[0]
stale_verdict_json = {
    'verdict': 'likely_true_positive',
    'confidence': 0.74,
    'severity': 'P1',
    'severity_rationale': 'Geo anomaly with intact MFA; threat-intel evidence is stale and cannot anchor a benign verdict.',
    'summary': 'User u_acct_lead authenticated from Bulgaria 30s after Portland session; threat_intel cached_at is 90 days old.',
    'attack_chain': [],
    'observed_facts': [
        {
            'fact_id': 'f1', 'claim': 'User has MFA enabled.',
            'retrieval_id': id_ref.retrieval_id,
            'field_path': 'mfa_enabled', 'expected_value': True, 'confidence': 0.95,
        },
        {
            'fact_id': 'f2', 'claim': 'Threat intel reputation is unknown.',
            'retrieval_id': ti_ref.retrieval_id,
            'field_path': 'reputation', 'expected_value': 'unknown', 'confidence': 0.45,
        },
    ],
    'inferences': [
        {
            'inference_id': 'i1',
            'claim': 'Threat intel reputation is too stale (cached 90d ago) to anchor a benign verdict, regardless of stated "unknown" status.',
            'supported_by_fact_ids': ['f2'],
            'confidence': 0.80,
        },
    ],
    'recommendations': [
        {
            'priority': 1, 'action': 'force_password_reset',
            'rationale': 'Defense in depth; threat-intel staleness blocks a likely_false_positive call.',
            'supported_by_inference_ids': ['i1'],
            'blast_radius': 'low', 'reversible': True, 'automatable': False,
        },
    ],
    'blast_radius': {'affected_assets': ['u_acct_lead']},
    'uncertainty': {'missing_enrichments': ['threat_intel_fresh']},
}
stale_client = SequenceClient([LLMResponse(
    content=json.dumps(stale_verdict_json),
    stop_reason='end_turn', tokens_in=2100, tokens_out=720, cost_usd=0.024, model='claude-sonnet-4-6',
)])
stale_response, stale_aug, stale_ext = reason(alert, plan, bundle, stale_client, sources=sources)
stale_outcome = validate_response(
    stale_response.content, stale_aug,
    triage_id='triage_nb_02', tenant_id=alert.tenant_id, alert_id=alert.alert_id,
    investigation_plan_dump=plan.model_dump(), received_at=alert.received_at,
    ai_metadata=AIMetadata(route_tier='standard_t2', model_chain=['sonnet'], cost_usd=0.024),
)
print(f'verdict:                       {stale_outcome.verdict.verdict}')
print(f'confidence:                    {stale_outcome.verdict.confidence}')
print(f'uncertainty.missing_enrichments: {stale_outcome.verdict.uncertainty.get("missing_enrichments", [])}')
print(f'validation failures:           {len(stale_outcome.failures)}')
print()
print("The verdict is NOT likely_false_positive — the agent treated stale-clean reputation as non-discriminating evidence.")

provider:            internal_reputation
provider_confidence: 0.45
cached_at:           2026-03-20 14:31:01.455207+00:00
reputation:          unknown

verdict:                       likely_true_positive
confidence:                    0.74
uncertainty.missing_enrichments: ['threat_intel_fresh']
validation failures:           0

The pipeline carried the stale/unknown threat-intel signature through to the verdict. The replayed T2 response landed at likely_true_positive (not likely_false_positive), with threat_intel_fresh listed in uncertainty.missing_enrichments.


## 3. Ransomware P0 with T3 Opus escalation

Escalation criteria: T2 returned low confidence (< 0.6) AND severity ∈ {P0, P1} AND family ∈ DEEP_FAMILIES.

Two clients exercise T3 below. The first uses `SequenceClient` with three illustrative responses so the self-consistency vote is visible (2 confirmed_TP vs 1 likely_TP → majority confirmed_TP). The second replays the real captured Opus response (`fixtures/llm_replays/cd8a1be0d7d1e45f1148e61c.json`, captured 2026-06-16T04:29:49Z; `live_api: true`). The replay uses `SequenceClient` rather than `FixtureReplayClient` because the T3 request payload includes runtime UUIDs that break digest matching — call-order replay sidesteps that without losing authenticity.

In [6]:
from pathlib import Path

rw_alert = CanonicalAlertEvent(
    tenant_id='tenant_a', alert_id='nb_demo_03', source_system='crowdstrike',
    source_adapter_version='crowdstrike_v1', rule_id='cs.ransomware.v2', rule_family='ransomware',
    received_at=datetime(2026, 6, 17, 14, 32, 11, tzinfo=UTC),
    detected_at=datetime(2026, 6, 17, 14, 32, 10, tzinfo=UTC),
    severity_hint='P0',
    primary_assets=[Asset(asset_id='srv_billing_01', asset_type='service', tenant_id='tenant_a')],
    summary='Mass file rename + entropy spike on billing host',
)
rw_plan = registry.build_plan('ransomware', 'P0')
rw_query = SourceQuery(tenant_id='tenant_a', alert_id=rw_alert.alert_id, entity_id='srv_billing_01', ioc=None, extra={'rule_family': 'ransomware'})
rw_bundle = run_fanout(rw_plan, rw_query, sources)

# T2 confidence of 0.4 is below the 0.6 escalation threshold; severity is P0;
# family is in DEEP_FAMILIES. All three conditions met -> escalate.
print(f'should_escalate(P0, ransomware, conf=0.4) -> {should_escalate("P0", "ransomware", 0.4)}')
print()

# Illustrative self-consistency vote: 3 sampled Opus calls, majority wins.
t3_client = SequenceClient([
    LLMResponse(content=json.dumps({'verdict': 'confirmed_true_positive'}), stop_reason='end_turn', tokens_in=2200, tokens_out=600, cost_usd=0.05, model='claude-opus-4-7'),
    LLMResponse(content=json.dumps({'verdict': 'confirmed_true_positive'}), stop_reason='end_turn', tokens_in=2200, tokens_out=600, cost_usd=0.05, model='claude-opus-4-7'),
    LLMResponse(content=json.dumps({'verdict': 'likely_true_positive'}), stop_reason='end_turn', tokens_in=2200, tokens_out=600, cost_usd=0.05, model='claude-opus-4-7'),
])
escalation = escalate_to_t3(rw_alert, rw_plan, rw_bundle, t3_client, sample_size=3)
print(f'illustrative majority verdict: {escalation.majority_verdict}')
print(f'illustrative cost:             ${escalation.cost_usd:.3f}')
print()

# Replay the real captured Opus response. The fixture's content is the actual
# Opus output; SequenceClient just hands it to the escalation function in
# place of a live call.
_fixture = json.loads(Path('fixtures/llm_replays/cd8a1be0d7d1e45f1148e61c.json').read_text())
live_response = LLMResponse(
    content=_fixture['content'],
    stop_reason=_fixture['stop_reason'],
    tool_calls=_fixture.get('tool_calls', []),
    tokens_in=_fixture['tokens_in'],
    tokens_out=_fixture['tokens_out'],
    cost_usd=_fixture['cost_usd'],
    model=_fixture['model'],
)
live_replay = SequenceClient([live_response])
live_escalation = escalate_to_t3(rw_alert, rw_plan, rw_bundle, live_replay, sample_size=1)
print(f'T3 live-captured majority verdict: {live_escalation.majority_verdict}')
print(f'T3 live-captured cost:             ${live_escalation.cost_usd:.5f}')
print(f'T3 live-captured model:            {live_escalation.sampled_at}')
print(f'T3 live-captured captured_at:      {_fixture["captured_at"]}')
print()
# Note: the live capture is a single sample, not a 3-sample vote — that's why
# its verdict (likely_TP) differs from the illustrative vote's majority
# (confirmed_TP). The live capture proves the engine actually calls real Opus;
# the illustrative vote demonstrates the self-consistency mechanism.

should_escalate(P0, ransomware, conf=0.4) -> True

illustrative majority verdict: confirmed_true_positive
illustrative cost:             $0.150

T3 live-captured majority verdict: likely_true_positive
T3 live-captured cost:             $0.18564
T3 live-captured model:            ['claude-opus-4-7']
T3 live-captured captured_at:      2026-06-16T04:29:49.411992+00:00



## 4. Live-API capture procedure (informational; do not run inline)

The fixture replayed in §3 was produced by `scripts/capture_t3_fixture.py`. To re-capture against the live API:

```bash
# Set ANTHROPIC_API_KEY in your shell or in .env at the repo root
uv run python scripts/capture_t3_fixture.py
```

The script reads the key from env or `.env`, fires one `_build_t3_request` call against the live API, and writes the response to `fixtures/llm_replays/<digest>.json` with `captured_at` + `live_api: true` markers plus token + latency metadata.

`uv run pytest` and `uv run eval` never make live API calls, so the repo remains reproducible without an Anthropic key.

## What this walkthrough demonstrated

- **Deterministic control plane.** T1 plan resolution is YAML lookup; routing, fan-out, and tier policy are code, not LLM decisions.
- **Grounded reasoning.** `observed_facts` cite real `retrieval_id` + `field_path` pairs, and the validator checks support against retrieval payloads.
- **Cost-aware retrieval.** Tier policy can refuse expensive sources on the default path; deeper retrieval is explicit, not automatic.
- **Failure-mode honesty.** Missing or stale evidence is surfaced in the verdict instead of hidden.
- **Bounded escalation.** T3 is reserved for low-confidence P0/P1 cases in deep families.
- **Reproducibility.** The notebook runs without an API key; the live Opus call is captured once and replayed.
